# Persona grants (R011, R066)

A plain SQL notebook, not a pipeline source file - grants are applied once
(and re-applied only when the policy changes), not recomputed on every run.

Five personas, five account-level groups (create these first in the Account
Console - User management - Groups, not the workspace admin settings - a
group defined in a workspace does not carry across workspaces):

- `data_engineers`  - builds/owns the pipeline, needs write access to bronze/silver/gold
- `data_analysts`   - R066: read-only, Gold only, nothing in bronze/silver/ops
- `data_managers`   - read-only, Gold only (country-level row filtering, if you
                       re-enable it later, narrows this further - that is a
                       separate, already-written notebook, not this one)
- `data_executives` - read-only, Gold + ops (governance/operational dashboards)
- `data_admins`     - full control, including managing grants themselves

The grain here is schema-level, not table-level - granting per-table for
dozens of tables is both harder to audit and easy to forget for a new table;
a new Gold table inherits the schema grant automatically, a new Bronze/Silver
table does not leak to analysts because they have no grant on those schemas
at all, not even USE SCHEMA.


In [ ]:
%sql
USE CATALOG maven_test;

## Catalog-level access

Every group needs `USE CATALOG` just to navigate into any schema at all -
this alone grants nothing to read or write; it is a prerequisite for every
other grant below, the same way a directory needs execute permission before
its contents are reachable.


In [ ]:
%sql
GRANT USE CATALOG ON CATALOG maven_test TO `data_engineers`;
GRANT USE CATALOG ON CATALOG maven_test TO `data_analysts`;
GRANT USE CATALOG ON CATALOG maven_test TO `data_managers`;
GRANT USE CATALOG ON CATALOG maven_test TO `data_executives`;
GRANT USE CATALOG ON CATALOG maven_test TO `data_admins`;

## Engineering: bronze + silver (build/own the pipeline)

Full read/write/create on the two layers nobody outside engineering should
be touching directly - Bronze is raw landed data, Silver is in-flight
cleaned data, neither is meant for business consumption (that is what Gold
summary tables and dashboards are for).


In [ ]:
%sql
GRANT USE SCHEMA ON SCHEMA maven_test.bronze TO `data_engineers`;
GRANT SELECT ON SCHEMA maven_test.bronze TO `data_engineers`;
GRANT MODIFY ON SCHEMA maven_test.bronze TO `data_engineers`;
GRANT CREATE TABLE ON SCHEMA maven_test.bronze TO `data_engineers`;
GRANT USE SCHEMA ON SCHEMA maven_test.silver TO `data_engineers`;
GRANT SELECT ON SCHEMA maven_test.silver TO `data_engineers`;
GRANT MODIFY ON SCHEMA maven_test.silver TO `data_engineers`;
GRANT CREATE TABLE ON SCHEMA maven_test.silver TO `data_engineers`;

## Gold: read access for everyone who consumes dashboards (R066)

`SELECT` only - no `MODIFY`, no `CREATE TABLE` - for analysts, managers and
executives; schema-level `SELECT` means a new Gold table (a new summary
table added later) is automatically readable without a fresh grant, closing
the gap R066 is actually worried about ("approved Gold data" stops meaning
anything if every new table needs a person to remember to grant it).

Engineers also get Gold read/write - they are the ones whose pipeline
populates these tables and who debug them when a dashboard shows something
wrong.


In [ ]:
%sql
GRANT USE SCHEMA ON SCHEMA maven_test.gold TO `data_analysts`;
GRANT SELECT ON SCHEMA maven_test.gold TO `data_analysts`;
GRANT USE SCHEMA ON SCHEMA maven_test.gold TO `data_managers`;
GRANT SELECT ON SCHEMA maven_test.gold TO `data_managers`;
GRANT USE SCHEMA ON SCHEMA maven_test.gold TO `data_executives`;
GRANT SELECT ON SCHEMA maven_test.gold TO `data_executives`;
GRANT USE SCHEMA ON SCHEMA maven_test.gold TO `data_engineers`;
GRANT SELECT ON SCHEMA maven_test.gold TO `data_engineers`;
GRANT MODIFY ON SCHEMA maven_test.gold TO `data_engineers`;
GRANT CREATE TABLE ON SCHEMA maven_test.gold TO `data_engineers`;

## Ops: governance/operational tables (R106's dashboard audience)

Read-only for executives (they are the audience for the Governance/
Operations dashboard) and engineers (they are the ones who act on a failed
pipeline run or a data-quality drop). Analysts and managers are deliberately
left out - `ops` is platform telemetry, not "approved Gold data" in R066's
sense, and over-granting it would blur that boundary.


In [ ]:
%sql
GRANT USE SCHEMA ON SCHEMA maven_test.ops TO `data_executives`;
GRANT SELECT ON SCHEMA maven_test.ops TO `data_executives`;
GRANT USE SCHEMA ON SCHEMA maven_test.ops TO `data_engineers`;
GRANT SELECT ON SCHEMA maven_test.ops TO `data_engineers`;

## Admin: full control

`data_admins` gets `ALL PRIVILEGES` at the catalog level, which cascades to
every schema and table beneath it, including ones created after this cell
runs - the one group meant to manage grants itself going forward, not just
read/write data.


In [ ]:
%sql
GRANT ALL PRIVILEGES ON CATALOG maven_test TO `data_admins`;

## Self-test (not part of any pipeline, run manually)

`SHOW GRANTS` on the catalog shows everything just applied, from your own
(presumably admin) account. As with the row-level-security notebook, this
only shows what YOU can see - to actually prove an analyst is blocked from
Bronze/Silver, a real member of `data_analysts` needs to log in and try a
query themselves.


In [ ]:
%sql
SHOW GRANTS ON CATALOG maven_test;

## Removing the grants (if you need to undo this)

Kept here for reference, commented out - uncomment and run only if you need
to roll the policy back.


In [ ]:
%sql
-- Uncomment only if you need to roll back the policy.
-- REVOKE USE CATALOG ON CATALOG maven_test FROM `data_engineers`;
-- REVOKE USE CATALOG ON CATALOG maven_test FROM `data_analysts`;
-- REVOKE USE CATALOG ON CATALOG maven_test FROM `data_managers`;
-- REVOKE USE CATALOG ON CATALOG maven_test FROM `data_executives`;
-- REVOKE USE CATALOG ON CATALOG maven_test FROM `data_admins`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.bronze FROM `data_engineers`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.silver FROM `data_engineers`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.gold FROM `data_analysts`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.gold FROM `data_managers`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.gold FROM `data_executives`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.gold FROM `data_engineers`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.ops FROM `data_executives`;
-- REVOKE ALL PRIVILEGES ON SCHEMA maven_test.ops FROM `data_engineers`;
-- REVOKE ALL PRIVILEGES ON CATALOG maven_test FROM `data_admins`;